# Part 1: Predicting House Prices with Linear Regression

**Goal:** Predict a house's sale price from its square footage and location (plus bedrooms and age).

**Model:** Linear Regression (scikit-learn)

**Why linear regression?** Price is a continuous number, and the main relationship we expect (bigger house → higher price) is roughly linear. Linear regression is also easy to interpret: each coefficient tells us how much the price changes when that feature changes by one unit.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_SEED = 42

## 2. Create the dataset

The starter example used only a handful of rows, which is too few to train and test a model meaningfully. I generated a synthetic dataset of 150 houses that follows realistic patterns:

- Each location has its own base price and price per square foot (Downtown is most expensive, Rural is least).
- Each bedroom adds about $8,000.
- Each year of age removes about $500 (depreciation).
- Random noise is added so the data isn't perfectly predictable, like a real market.

A fixed random seed makes the results reproducible.

In [2]:
def generate_house_data(n_records=150):
    rng = np.random.default_rng(RANDOM_SEED)
    locations = ["Downtown", "Suburb", "Rural", "Uptown"]
    price_per_sqft = {"Downtown": 350, "Uptown": 275, "Suburb": 200, "Rural": 120}
    base_price = {"Downtown": 150_000, "Uptown": 100_000, "Suburb": 60_000, "Rural": 30_000}

    location = rng.choice(locations, size=n_records, p=[0.25, 0.25, 0.3, 0.2])
    square_footage = rng.normal(1800, 550, size=n_records).clip(600, 5000).round(0)
    bedrooms = rng.integers(1, 6, size=n_records)
    age_years = rng.integers(0, 60, size=n_records)

    price = np.array([
        base_price[loc] + price_per_sqft[loc] * sqft + 8_000 * bed - 500 * age + rng.normal(0, 15_000)
        for loc, sqft, bed, age in zip(location, square_footage, bedrooms, age_years)
    ]).clip(min=40_000).round(2)

    return pd.DataFrame({"square_footage": square_footage, "bedrooms": bedrooms,
                         "age_years": age_years, "location": location, "price": price})

df = generate_house_data()
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (150, 5)


,square_footage,bedrooms,age_years,location,price
0,2386.0,3,39,Rural,325749.93
1,1886.0,3,59,Suburb,437128.83
2,1713.0,1,0,Uptown,598884.92
3,1230.0,3,43,Rural,174958.21
4,879.0,4,50,Downtown,442497.13


In [3]:
df.groupby("location")["price"].agg(["count", "mean"]).round(0).sort_values("mean", ascending=False)

,count,mean
location,,
Downtown,37,750245.0
Uptown,21,619352.0
Suburb,44,426710.0
Rural,48,262944.0


The average price by location confirms the pattern built into the data: Downtown > Uptown > Suburb > Rural.

## 3. Prepare features and split into train / test sets

`location` is text, so the model can't use it directly. **One-hot encoding** turns it into 0/1 columns (one per location). I use `drop="first"` so one location (Downtown) becomes the baseline and the other coefficients are measured relative to it.

I hold out 20% of the data as a test set so the model is evaluated on houses it has never seen.

In [4]:
X = df[["square_footage", "bedrooms", "age_years", "location"]]
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

preprocessor = ColumnTransformer(
    transformers=[("location_ohe", OneHotEncoder(drop="first"), ["location"])],
    remainder="passthrough",
)

model = Pipeline([("preprocess", preprocessor), ("regressor", LinearRegression())])
model.fit(X_train, y_train)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

Training rows: 120 | Test rows: 30


## 4. Evaluate the model

- **R²** is the share of the variation in price that the model explains (1.0 = perfect).
- **RMSE** is the typical size of a prediction error, in dollars.

In [5]:
y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)
print(f"Test R^2:  {r2:.3f}")
print(f"Test RMSE: ${rmse:,.0f}")

Test R^2:  0.966
Test RMSE: $43,600


**Interpretation:** The model explains about 97% of the variation in price on unseen houses. A typical prediction is off by roughly $44k, which is reasonable given that prices in this data range from under $200k to over $1M.

## 5. Predict the price of a new house

In [6]:
new_house = pd.DataFrame({"square_footage": [2000], "bedrooms": [3], "age_years": [10], "location": ["Downtown"]})
print(f"Predicted price (2,000 sq ft, 3 bed, 10 yrs old, Downtown): ${model.predict(new_house)[0]:,.0f}")

Predicted price (2,000 sq ft, 3 bed, 10 yrs old, Downtown): $827,031


## 6. Interpret the coefficients

In [7]:
ohe = model.named_steps["preprocess"].named_transformers_["location_ohe"]
feature_names = list(ohe.get_feature_names_out(["location"])) + ["square_footage", "bedrooms", "age_years"]
coefs = pd.Series(model.named_steps["regressor"].coef_, index=feature_names).round(2)
coefs.to_frame("coefficient")

,coefficient
location_Rural,-518990.90
location_Suburb,-338655.03
location_Uptown,-165136.25
square_footage,224.67
bedrooms,10654.60
age_years,-640.42


**What the coefficients mean:**

- **square_footage:** each extra square foot adds about $225 to the predicted price.
- **bedrooms:** each extra bedroom adds about $10.6k (close to the $8k built into the data).
- **age_years:** each year of age lowers the price by about $640 (close to the $500 built in).
- **location_***: compared with Downtown, Rural, Suburb and Uptown houses are all predicted to be cheaper, with Rural the cheapest.

## 7. Limitations

- In this data, each location has its **own** price per square foot, but a plain linear regression fits **one** slope for everyone. The model makes up for this with very large location coefficients (e.g., Rural ≈ −$519k). Adding a `square_footage × location` interaction term would fit this better.
- The data is synthetic, so the model is recovering patterns I built in. On a real dataset I would expect lower accuracy and more features to matter (lot size, condition, school district, etc.).